# Polynomial Regression: let a line bend

**Goal:** fit a curved pattern when a straight line misses the dots.

We give the model extra clues such as x². Linear Regression learns how to combine these clues. The result can curve even though the model is still linear in its weights.

## 1. Bring in the tools

NumPy makes examples, Matplotlib draws them, and scikit-learn builds the extra clues and learns the model.

In [ ]:
import numpy as np  # Make input values and random noise.
import matplotlib.pyplot as plt  # Draw dots and curves.

from sklearn.model_selection import train_test_split  # Save unseen rows for a test.
from sklearn.pipeline import make_pipeline  # Join feature-making and regression.
from sklearn.preprocessing import PolynomialFeatures  # Create x-squared and higher powers.
from sklearn.linear_model import LinearRegression  # Learn weights for those clues.
from sklearn.metrics import mean_absolute_error, r2_score  # Check mistakes and fit.

## 2. Make a pretend U-shaped pattern

Imagine x is a machine setting and y is its measured output. We make a U-shape on purpose. A little random noise makes the dots less perfect, like real measurements.

In [ ]:
rng = np.random.default_rng(7)  # Make the same noise each time.
X = np.linspace(0, 8, 40).reshape(-1, 1)  # Make 40 inputs as a one-column table.
x = X[:, 0]  # Get the values from that column.
noise = rng.normal(0, 1.2, size=len(X))  # Add small random changes.
y = 0.45 * (x - 4) ** 2 + 2 + noise  # Make a U-shape plus noise.

print("Input shape:", X.shape)  # Rows first, then input columns.
print("First five answers:", y[:5].round(1))  # Preview five answers.

**Read the data code:** the formula makes y small near x = 4 and larger farther away, so the dots make a U. X stays a table because scikit-learn expects rows and columns.

## 3. Look at the dots

A straight line works when dots roughly follow a straight path. These dots bend, so a straight line may miss their shape.

In [ ]:
plt.scatter(X[:, 0], y, color="steelblue")  # Draw one dot for every example.
plt.xlabel("Input x")  # Name the horizontal axis.
plt.ylabel("Answer y")  # Name the vertical axis.
plt.title("The examples bend into a U-shape")
plt.grid(alpha=0.25)
plt.show()

## 4. Keep some dots hidden for testing

The model learns from training dots. We hide test dots until the end to see how it handles examples it did not learn from.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)  # Learn from 75%; keep 25% hidden.

print("Training dots:", len(X_train))
print("Hidden test dots:", len(X_test))

## 5. Add a curved clue

For x = 3, degree 2 gives two clues: x = 3 and x² = 9. PolynomialFeatures makes the extra clue; LinearRegression learns how much each clue matters.

A pipeline connects both steps, so training, testing, and new inputs get the same feature recipe.

In [ ]:
degree = 2  # Ask for x and x-squared clues.
curve_model = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),  # Make the clues.
    LinearRegression()  # Learn how much each clue matters.
)
curve_model.fit(X_train, y_train)  # Learn using training dots only.
print("For x=3, degree-2 clues:", curve_model[0].transform([[3]])[0])

**Why include_bias=False?** It skips an extra column of ones. LinearRegression already learns a starting value, called the intercept.

## 6. Compare a straight line and two curves

Degree 1 is a straight line. Degree 2 can make a smooth U-shape. A high degree can wiggle to match noise and perform poorly on new examples; this is overfitting.

The test scores here are for learning. In a real project, choose degree with validation or cross-validation, then use a separate test set once for the final check.

In [ ]:
degrees = [1, 2, 8]  # Compare a line, a simple curve, and a bendy curve.
models = {}  # Save each trained model for the graph.
score_rows = []  # Save each model's results.

for degree in degrees:  # Repeat these steps for each degree.
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),  # Make powers of x.
        LinearRegression()  # Learn weights for those powers.
    )
    model.fit(X_train, y_train)  # Learn from training examples only.
    train_guess = model.predict(X_train)  # Check guesses on training examples.
    test_guess = model.predict(X_test)  # Check guesses on hidden examples.
    models[degree] = model  # Keep the trained model for later.
    score_rows.append({
        "degree": degree,
        "train_MAE": mean_absolute_error(y_train, train_guess),
        "test_MAE": mean_absolute_error(y_test, test_guess),
        "train_R2": r2_score(y_train, train_guess),
        "test_R2": r2_score(y_test, test_guess)
    })  # Store the mistake sizes and R-squared scores.

score_rows  # Show the comparison.

### Read the comparison

MAE is the average size of a mistake, so smaller is better. R-squared closer to 1 usually means a better fit. If a very bendy model does much better on training but worse on test examples, it may have memorized the training dots. A high degree is not automatically bad; check unseen data.

## 7. Draw each fitted shape

The dots are examples. Each colored line is one model's guess across many x-values. Higher degree allows more bending.

In [ ]:
x_line = np.linspace(X.min(), X.max(), 250).reshape(-1, 1)  # Make points for smooth curves.

plt.scatter(X_train[:, 0], y_train, color="steelblue", label="Training examples")
plt.scatter(X_test[:, 0], y_test, color="black", marker="x", label="Hidden test examples")

for degree, model in models.items():  # Use each trained model.
    y_line = model.predict(x_line)  # Pipeline makes powers and predicts.
    plt.plot(x_line[:, 0], y_line, label=f"Degree {degree}")

plt.xlabel("Input x")
plt.ylabel("Answer y")
plt.title("Higher degree allows more bending")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 8. Predict for a new input

Give the pipeline one new x in the same one-column table shape. The pipeline creates the extra powers before making its guess.

In [ ]:
new_x = np.array([[6]])  # One new input, as a one-row table.
new_answer = models[2].predict(new_x)[0]  # Use the degree-2 pipeline.
print(f"For x=6, predicted y = {new_answer:.2f}")

## Quick revision

- Use Polynomial Regression when a straight line misses a curve.
- Degree 2 gives x and x²; degree 3 adds x³.
- PolynomialFeatures creates clues; LinearRegression learns their weights.
- Train on training data and check on unseen data.
- Too little bending can underfit; too much can overfit.
- A pipeline applies the same feature recipe to train, test, and new inputs.
- Choose degree with validation or cross-validation.

**Remember:** make powers of x → learn their weights → check on unseen examples.